In [ ]:
# Imports

import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

# Load data

data = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/d51iMGfp_t0QpO30Lym-dw/automobile-sales.csv')

# Create app

app = dash.Dash(__name__)

# Title

app.title = "Automobile Sales Statistics Dashboard"

# Year list

year_list = [i for i in range(1980, 2024)]

# =====================================
# LAYOUT
# =====================================

app.layout = html.Div([

    html.H1(
        "Automobile Sales Statistics Dashboard",
        style={
            'textAlign': 'center',
            'color': '#503D36',
            'fontSize': 24
        }
    ),

    dcc.Dropdown(
        id='dropdown-statistics',
        options=[
            {
                'label': 'Yearly Statistics',
                'value': 'Yearly Statistics'
            },
            {
                'label': 'Recession Period Statistics',
                'value': 'Recession Period Statistics'
            }
        ],
        placeholder='Select a report type'
    ),

    dcc.Dropdown(
        id='select-year',
        options=[
            {'label': i, 'value': i}
            for i in year_list
        ],
        placeholder='Select-year'
    ),

    html.Div(
        id='output-container',
        className='chart-grid'
    )

])

# =====================================
# CALLBACK 1
# Enable/Disable year dropdown
# =====================================
@app.callback(
    Output('select-year', 'disabled'),
    Input('dropdown-statistics', 'value')
)
def update_year_dropdown(selected_statistics):

    if selected_statistics == 'Yearly Statistics':
        return False
    return True
@app.callback(
    Output('output-container', 'children'),
    [
        Input('dropdown-statistics', 'value'),
        Input('select-year', 'value')
    ]
)
def update_output_container(selected_statistics, input_year):

    # ---------------- RECESSION ----------------
    if selected_statistics == 'Recession Period Statistics':

        recession_data = data[data['Recession'] == 1]

        yearly_rec = recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()

        R_chart1 = dcc.Graph(
            figure=px.line(
                yearly_rec,
                x='Year',
                y='Automobile_Sales',
                title='Average Automobile Sales during Recession'
            )
        )

        average_sales = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()

        R_chart2 = dcc.Graph(
            figure=px.bar(
                average_sales,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Vehicles Sold by Type (Recession)'
            )
        )

        exp_rec = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()

        R_chart3 = dcc.Graph(
            figure=px.pie(
                exp_rec,
                names='Vehicle_Type',
                values='Advertising_Expenditure',
                title='Advertising Share by Vehicle Type (Recession)'
            )
        )

        unemp_data = recession_data.groupby(
            ['Vehicle_Type', 'unemployment_rate']
        )['Automobile_Sales'].mean().reset_index()

        R_chart4 = dcc.Graph(
            figure=px.bar(
                unemp_data,
                x='Vehicle_Type',
                y='Automobile_Sales',
                color='unemployment_rate',
                title='Effect of Unemployment Rate on Sales (Recession)'
            )
        )

        return [
            html.Div([R_chart1, R_chart2], style={'display': 'flex'}),
            html.Div([R_chart3, R_chart4], style={'display': 'flex'})
        ]


    # ---------------- YEARLY ----------------
    elif selected_statistics == 'Yearly Statistics':

        yearly_data = data[data['Recession'] == 0]

        y_sales = yearly_data.groupby('Year')['Automobile_Sales'].mean().reset_index()

        Y_chart1 = dcc.Graph(
            figure=px.line(
                y_sales,
                x='Year',
                y='Automobile_Sales',
                title='Yearly Automobile Sales Trend'
            )
        )

        y_vehicle = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()

        Y_chart2 = dcc.Graph(
            figure=px.bar(
                y_vehicle,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Sales by Vehicle Type (Yearly)'
            )
        )

        y_exp = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()

        Y_chart3 = dcc.Graph(
            figure=px.pie(
                y_exp,
                names='Vehicle_Type',
                values='Advertising_Expenditure',
                title='Advertising Share by Vehicle Type (Yearly)'
            )
        )

        y_unemp = yearly_data.groupby(
            ['Vehicle_Type', 'unemployment_rate']
        )['Automobile_Sales'].mean().reset_index()

        Y_chart4 = dcc.Graph(
            figure=px.bar(
                y_unemp,
                x='Vehicle_Type',
                y='Automobile_Sales',
                color='unemployment_rate',
                title='Unemployment Effect on Sales (Yearly)'
            )
        )

        return [
            html.Div([Y_chart1, Y_chart2], style={'display': 'flex'}),
            html.Div([Y_chart3, Y_chart4], style={'display': 'flex'})
        ]

    return []

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=8050)